# Nexora Care Flow: Appointment Volume Forecasting
## Step 1: Data Cleaning

In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("../data/AppointmentRecords.csv")

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129353 entries, 0 to 129352
Data columns (total 15 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   AppointmentID         129353 non-null  int64 
 1   ClinicID              129353 non-null  int64 
 2   ClinicName            129353 non-null  object
 3   ClinicType            129353 non-null  object
 4   PatientID             129353 non-null  int64 
 5   ProviderID            129353 non-null  int64 
 6   AppointmentDateTime   129353 non-null  object
 7   AppointmentType       129353 non-null  object
 8   Status                129353 non-null  object
 9   BookingChannel        129353 non-null  object
 10  BookingLeadTimeDays   129353 non-null  int64 
 11  IsHoliday             129353 non-null  bool  
 12  HolidayName           939 non-null     object
 13  SeasonFlag            129353 non-null  object
 14  ChronicConditionFlag  129353 non-null  bool  
dtypes: bool(2), int64

In [5]:
df.head()

,AppointmentID,ClinicID,ClinicName,ClinicType,PatientID,ProviderID,AppointmentDateTime,AppointmentType,Status,BookingChannel,BookingLeadTimeDays,IsHoliday,HolidayName,SeasonFlag,ChronicConditionFlag
0,529385,2,Nexora Care - Lakeside,Multi-Specialty,103642,202,2024-01-01 08:00,New Patient,Completed,Online,2,True,New Year's Day,Flu Season,False
1,500000,1,Nexora Care - Riverside,Primary Care,101551,101,2024-01-01 08:30,New Patient,No-Show,Online,18,True,New Year's Day,Flu Season,False
2,500011,1,Nexora Care - Riverside,Primary Care,100260,104,2024-01-01 08:30,Follow-Up,Completed,Referral,4,True,New Year's Day,Flu Season,False
3,575066,3,Nexora Care - Downtown Express,Telehealth-Forward,104923,301,2024-01-01 09:45,Urgent,Completed,Online,0,True,New Year's Day,Flu Season,False
4,500008,1,Nexora Care - Riverside,Primary Care,101124,104,2024-01-01 09:45,Telehealth,Completed,Phone,2,True,New Year's Day,Flu Season,False


In [6]:
df.shape

(129353, 15)

# Missing Values

In [7]:
df.isnull().sum()

AppointmentID                0
ClinicID                     0
ClinicName                   0
ClinicType                   0
PatientID                    0
ProviderID                   0
AppointmentDateTime          0
AppointmentType              0
Status                       0
BookingChannel               0
BookingLeadTimeDays          0
IsHoliday                    0
HolidayName             128414
SeasonFlag                   0
ChronicConditionFlag         0
dtype: int64

In [8]:
pd.crosstab(df["IsHoliday"], df["HolidayName"].notnull())

HolidayName,False,True
IsHoliday,,
False,128414,0
True,0,939


In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
df["AppointmentID"].duplicated().sum()

np.int64(0)

# Keep a Raw Copy

In [11]:
df_raw = df.copy()

# Summary Statistics and Unique Values

In [12]:
df.describe()

,AppointmentID,ClinicID,PatientID,ProviderID,BookingLeadTimeDays
count,129353.000000,129353.000000,129353.000000,129353.000000,129353.000000
mean,564676.000000,2.417864,103553.485385,245.290005,8.704267
std,37341.139023,1.071525,2020.941152,107.165883,7.641921
min,500000.000000,1.000000,100000.000000,101.000000,0.000000
25%,532338.000000,2.000000,101812.000000,201.000000,3.000000
50%,564676.000000,2.000000,103547.000000,205.000000,7.000000
75%,597014.000000,3.000000,105178.000000,306.000000,12.000000
max,629352.000000,4.000000,107279.000000,406.000000,60.000000


In [13]:
df.nunique()

AppointmentID           129353
ClinicID                     4
ClinicName                   4
ClinicType                   3
PatientID                 7109
ProviderID                  24
AppointmentDateTime      30287
AppointmentType              4
Status                       4
BookingChannel               4
BookingLeadTimeDays         61
IsHoliday                    2
HolidayName                 13
SeasonFlag                   3
ChronicConditionFlag         2
dtype: int64

In [14]:
df.dtypes

AppointmentID            int64
ClinicID                 int64
ClinicName              object
ClinicType              object
PatientID                int64
ProviderID               int64
AppointmentDateTime     object
AppointmentType         object
Status                  object
BookingChannel          object
BookingLeadTimeDays      int64
IsHoliday                 bool
HolidayName             object
SeasonFlag              object
ChronicConditionFlag      bool
dtype: object

# Categorical Values and Anomalies

In [15]:
for col in ["ClinicName", "ClinicType", "Status", "AppointmentType", "BookingChannel", "SeasonFlag"]:
    print(df[col].value_counts(), "\n")

ClinicName
Nexora Care - Lakeside            45681
Nexora Care - Riverside           29382
Nexora Care - Hillcrest           29144
Nexora Care - Downtown Express    25146
Name: count, dtype: int64 

ClinicType
Primary Care          58526
Multi-Specialty       45681
Telehealth-Forward    25146
Name: count, dtype: int64 

Status
Completed      109040
No-Show         11367
Cancelled        6296
Rescheduled      2650
Name: count, dtype: int64 

AppointmentType
Follow-Up      57859
Telehealth     30117
New Patient    21583
Urgent         19794
Name: count, dtype: int64 

BookingChannel
Online      57061
Phone       39273
Referral    19394
Walk-In     13625
Name: count, dtype: int64 

SeasonFlag
Standard          64015
Flu Season        34614
Allergy Season    30724
Name: count, dtype: int64 



In [16]:
pd.crosstab(df["ClinicID"], df["ClinicName"])

ClinicName,Nexora Care - Downtown Express,Nexora Care - Hillcrest,Nexora Care - Lakeside,Nexora Care - Riverside
ClinicID,,,,
1,0,0,0,29382
2,0,0,45681,0
3,25146,0,0,0
4,0,29144,0,0


In [17]:
df["AppointmentDateTime"].str.len().value_counts()

AppointmentDateTime
16    129353
Name: count, dtype: int64

In [18]:
df["AppointmentDateTime"] = pd.to_datetime(df["AppointmentDateTime"], errors="coerce")

In [19]:
df["AppointmentDateTime"].isnull().sum()

np.int64(0)

In [20]:
df["AppointmentDateTime"].min(), df["AppointmentDateTime"].max()

(Timestamp('2024-01-01 08:00:00'), Timestamp('2025-12-31 17:45:00'))

In [21]:
df["BookingChannel"].value_counts()

BookingChannel
Online      57061
Phone       39273
Referral    19394
Walk-In     13625
Name: count, dtype: int64

In [22]:
df["SeasonFlag"].value_counts()

SeasonFlag
Standard          64015
Flu Season        34614
Allergy Season    30724
Name: count, dtype: int64

# Clinic Closures and Daily Coverage

In [23]:
df["Date"] = df["AppointmentDateTime"].dt.normalize()

In [24]:
df["Date"].nunique()

731

In [26]:
df.groupby("ClinicID")["Date"].nunique()

ClinicID
1    627
2    627
3    731
4    472
Name: Date, dtype: int64

Duplicates excluding AppointmentID

In [27]:
cols = [c for c in df.columns if c != "AppointmentID"]

In [28]:
df.duplicated(subset=cols).sum()

np.int64(1)

In [29]:
df[df.duplicated(subset=cols, keep=False)].sort_values(cols).head(10)

,AppointmentID,ClinicID,ClinicName,ClinicType,PatientID,ProviderID,AppointmentDateTime,AppointmentType,Status,BookingChannel,BookingLeadTimeDays,IsHoliday,HolidayName,SeasonFlag,ChronicConditionFlag,Date
116139,625580,4,Nexora Care - Hillcrest,Primary Care,106014,401,2025-10-29 10:30:00,Urgent,Completed,Phone,1,False,NaN,Standard,False,2025-10-29
116143,625614,4,Nexora Care - Hillcrest,Primary Care,106014,401,2025-10-29 10:30:00,Urgent,Completed,Phone,1,False,NaN,Standard,False,2025-10-29


In [30]:
rows_before = len(df)

In [31]:
df = df.drop_duplicates(subset=cols, keep="first")

In [32]:
rows_before, len(df)

(129353, 129352)

In [33]:
df.groupby("ClinicID")["Date"].agg(["min", "max", "nunique"])

,min,max,nunique
ClinicID,,,
1,2024-01-01,2025-12-31,627
2,2024-01-01,2025-12-31,627
3,2024-01-01,2025-12-31,731
4,2024-06-29,2025-12-31,472


In [34]:
df["DayOfWeek"] = df["AppointmentDateTime"].dt.day_name()

In [35]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
pd.crosstab(df["ClinicID"], df["DayOfWeek"])[day_order]

DayOfWeek,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Sunday
ClinicID,,,,,,,
1,5324,6030,5739,5411,5044,1834,0
2,7823,9063,8491,8240,7445,4619,0
3,3867,4296,4104,3898,3770,3000,2211
4,5234,5961,5856,5432,5056,1604,0


check for unusually low days

In [36]:
daily = df.groupby(["ClinicID", "Date"]).size().reset_index(name="Appointments")

In [37]:
daily.groupby("ClinicID")["Appointments"].describe()

,count,mean,std,min,25%,50%,75%,max
ClinicID,,,,,,,,
1,627.0,46.861244,19.564056,2.0,37.00,51.0,60.0,99.0
2,627.0,72.856459,24.005826,7.0,59.00,75.0,90.0,140.0
3,731.0,34.399453,12.184841,1.0,27.00,35.0,43.0,84.0
4,472.0,61.743644,26.458315,7.0,47.75,68.0,80.0,126.0


In [38]:
daily.sort_values("Appointments").head(10)

,ClinicID,Date,Appointments
1765,3,2025-05-26,1
210,1,2024-09-02,2
583,1,2025-11-11,3
1401,3,2024-05-27,3
1587,3,2024-11-29,3
1934,3,2025-11-11,3
1303,3,2024-02-19,4
1268,3,2024-01-15,4
1804,3,2025-07-04,4
1951,3,2025-11-28,4


In [39]:
daily_low = daily.sort_values("Appointments").head(10)
df[df["Date"].isin(daily_low["Date"])].groupby("Date")[["IsHoliday", "HolidayName"]].first()

,IsHoliday,HolidayName
Date,,
2024-01-15,True,MLK Day
2024-02-19,True,Presidents' Day
2024-05-27,True,Memorial Day
2024-09-02,True,Labor Day
2024-11-29,True,Day After Thanksgiving
2025-05-26,True,Memorial Day
2025-07-04,True,Independence Day
2025-11-11,True,Veterans Day
2025-11-28,True,Day After Thanksgiving


# Data-Quality Log

In [41]:
dq_log = pd.DataFrame({
    "Check": [
        "Missing values",
        "HolidayName nulls",
        "Exact duplicate rows",
        "Duplicate AppointmentID",
        "Near-duplicate (ignoring ID)",
        "Date format",
        "Date range",
        "Categorical spellings",
        "Lead time values",
        "Clinic closures",
        "Quiet days",
    ],
    "Finding": [
        "None in any column except HolidayName",
        "128,414 empty, matches IsHoliday = False exactly",
        "0 found",
        "0 found",
        "1 pair found: IDs 625580 and 625614 (Hillcrest, 2025-10-29 10:30)",
        "All 129,353 values in one format, no conversion failures",
        "2024-01-01 to 2025-12-31 (brief said one quarter)",
        "No variants in clinic, status, type, channel or season",
        "0 to 60 days, no negatives",
        "Sundays closed at clinics 1, 2 and 4; Hillcrest opened 2024-06-29",
        "Lowest days are public holidays, not zero-volume closures",
    ],
    "Action": [
        "None needed",
        "Kept as is, valid nulls",
        "None needed",
        "None needed",

        "Removed 625614, kept 625580",
        "Converted to datetime",
        "Documented difference from the brief",
        "None needed",
        "None needed",
        "Closed days left out of daily counts, not filled with zeros",
        "Kept, real holiday demand dips",
    ],
})
dq_log

,Check,Finding,Action
0,Missing values,None in any column except HolidayName,None needed
1,HolidayName nulls,"128,414 empty, matches IsHoliday = False exactly","Kept as is, valid nulls"
2,Exact duplicate rows,0 found,None needed
3,Duplicate AppointmentID,0 found,None needed
4,Near-duplicate (ignoring ID),1 pair found: IDs 625580 and 625614 (Hillcrest...,"Removed 625614, kept 625580"
5,Date format,"All 129,353 values in one format, no conversio...",Converted to datetime
6,Date range,2024-01-01 to 2025-12-31 (brief said one quarter),Documented difference from the brief
7,Categorical spellings,"No variants in clinic, status, type, channel o...",None needed
8,Lead time values,"0 to 60 days, no negatives",None needed
9,Clinic closures,"Sundays closed at clinics 1, 2 and 4; Hillcres...","Closed days left out of daily counts, not fill..."


# Export the Clean Dataset

In [42]:
df.to_csv("../data/AppointmentRecords_clean.csv", index=False)

In [43]:
dq_log.to_csv("../data/data_quality_log.csv", index=False)

In [44]:
df.shape

(129352, 17)